In [1]:
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

import SCRIPTS.jsonDownloader as jd
from SCRIPTS.redditLinkRetriever import fetch_saved_post_links, save_links_txt
from SCRIPTS.mediaDownloader import download_embedded_media
from SCRIPTS.mediaOrganizer import organize_downloads
from SCRIPTS.redgifDownloader import process_external
from SCRIPTS.cloudflareUploader import upload_media
from SCRIPTS.r2_audit import audit_local_vs_r2

In [2]:
# https://old.reddit.com/prefs/apps
# Set to True to avoid making any changes
DRY_RUN_MEDIA = False
DRY_RUN_ORGANIZE = False

DRY_RUN_CLOUDFLARE = False   # True = preview only, no upload
ACCOUNT_INDEX = 2            # choose which R2 account (1, 2, ...)
CHECK_ONLY = False           # True = just check existence, no upload

DRY_RUN_FINAL = False

In [3]:
# Retrieve saved post links for the specified user
links = fetch_saved_post_links()

Version 7.8.1 of praw is outdated. Version 8.0.2 was released Wednesday June 24, 2026.


In [4]:
len(links)

334

In [5]:
links[:5]

['https://www.reddit.com/r/IWantToBeHerHentai2/comments/1uqmt2x/i_wanna_turn_myself_into_nothing_but_a_handy',
 'https://www.reddit.com/r/bangmybully/comments/1unq6um/my_boyfriend_is_such_an_idiot',
 'https://www.reddit.com/r/IWantToBeHerHentai2/comments/1uo00u7/i_want_to_accidentally_send_a_picture_like_this',
 'https://www.reddit.com/r/rape_hentai/comments/1hvejqo/ffuck_hes_using_me_like_a_fleshlight',
 'https://www.reddit.com/r/rape_hentai/comments/1ujuqky/so_this_is_why_cheerleaders_use_those_short']

# NEW POST VALIDATION

This section validates new posts from reddits saved folder

In [6]:
csv_path = Path("ordered_posts.csv")
raw_df = pd.read_csv(csv_path)

POST_ID_RE = re.compile(r"/comments/([a-z0-9]+)(?:[/?#]|$)", re.IGNORECASE)
SHORT_RE   = re.compile(r"redd\.it/([a-z0-9]+)(?:[/?#]|$)", re.IGNORECASE)
max_order_num = raw_df.order_num.max()

def strip_trailing_slash(url: str) -> str:
    # remove trailing slashes only at the very end (doesn't touch scheme)
    return url.rstrip("/")

def extract_post_id(url: str) -> str | None:
    """
    Try to extract a post id from:
      - standard permalink: .../comments/<postid>/...
      - shortlink: https://redd.it/<postid>
    """
    m = POST_ID_RE.search(url)
    if m:
        return m.group(1)
    m = SHORT_RE.search(url)
    if m:
        return m.group(1)
    return None

existing_ids = set(str(x).lower() for x in raw_df.get("post_id", pd.Series([])).dropna())

new_rows = []
next_order = max_order_num + 1
seen_in_batch = set()  # avoid duplicates within this run

for raw_link in reversed(links):
    link = strip_trailing_slash(raw_link)
    post_id = extract_post_id(link)
    if not post_id:
        continue
    pid = post_id.lower()

    # Only add if NOT already in CSV and not already queued this batch
    if pid in existing_ids or pid in seen_in_batch:
        continue

    new_rows.append({
        "order_num": next_order,
        "link": link,
        "post_id": post_id,
        "date_added": datetime.utcnow().isoformat(timespec="seconds"),
    })
    seen_in_batch.add(pid)
    next_order += 1

# Preview as a DataFrame
new_df = pd.DataFrame(new_rows)
new_df


C:\Users\minds\AppData\Local\Temp\ipykernel_32128\787629229.py:47: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date_added": datetime.utcnow().isoformat(timespec="seconds"),


,order_num,link,post_id,date_added
0,1732,https://www.reddit.com/r/AhegaoGirls/comments/...,1uf8rr0,2026-07-10T00:43:41
1,1733,https://www.reddit.com/r/bangmybully/comments/...,1ubz3gl,2026-07-10T00:43:41
2,1734,https://www.reddit.com/r/Rapekink/comments/1uh...,1uhzwwj,2026-07-10T00:43:41
3,1735,https://www.reddit.com/r/HentaiBullying/commen...,1uh04dd,2026-07-10T00:43:41
4,1736,https://www.reddit.com/r/HentaiBeast/comments/...,1ugz9ws,2026-07-10T00:43:41
5,1737,https://www.reddit.com/r/RapeR34/comments/1uix...,1uixwfb,2026-07-10T00:43:41
6,1738,https://www.reddit.com/r/rape_hentai/comments/...,1ujuqky,2026-07-10T00:43:41
7,1739,https://www.reddit.com/r/rape_hentai/comments/...,1hvejqo,2026-07-10T00:43:41
8,1740,https://www.reddit.com/r/IWantToBeHerHentai2/c...,1uo00u7,2026-07-10T00:43:41
9,1741,https://www.reddit.com/r/bangmybully/comments/...,1unq6um,2026-07-10T00:43:41


In [7]:
final_df = pd.concat([raw_df, new_df], ignore_index=True)
final_df = final_df.sort_values(by="order_num", ascending=False).reset_index(drop=True)

In [8]:
import importlib
importlib.reload(jd)

jd.configure(
    DATA_ROOT="Out",
    SKIP_EXISTING=False,
    REPORTS_DIR="__reports",
    write_csv_to=None
    )

summary = jd.process_all(new_df["link"].tolist(), show_progress=True)
summary

  0%|          | 0/11 [00:00<?, ?post/s]

Done. Success: 11, Skipped: 0, Failed: 0


{'success': 11, 'skipped': 0, 'failed': 0}

# MEDIA DOWNLOADER
Reviews the external and media json folders in **Out/**, downloading:
- Images
- Gifs
- Videos

In [9]:
folders = ["external", "media"]
download_stats = []

# point to your inputs/outputs explicitly
for mediaType in folders:
    download_stats.append(download_embedded_media(
        media_json_dir=Path("Out/" + mediaType),   # where your *.json live
        media_out_dir=Path("Media/media_files"),  # where downloads should go
        write_fail_csv_to=Path("__reports/media_report" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
        show_progress=True,
    ))

download_stats

[{'downloaded': 0,
  'failed': 49,
  'skipped': 0,
  'fail_rows': [{'id': '1hvejqo', 'reason': 'no_reddit_media_url'},
   {'id': '1oyhfm4', 'reason': 'no_reddit_media_url'},
   {'id': '1q6n5qv', 'reason': 'no_reddit_media_url'},
   {'id': '1r40e0r', 'reason': 'no_reddit_media_url'},
   {'id': '1rg3fpf', 'reason': 'no_reddit_media_url'},
   {'id': '1rg5m46', 'reason': 'no_reddit_media_url'},
   {'id': '1rgmzbe', 'reason': 'no_reddit_media_url'},
   {'id': '1rheclu', 'reason': 'no_reddit_media_url'},
   {'id': '1ri39na', 'reason': 'no_reddit_media_url'},
   {'id': '1rngr2r', 'reason': 'no_reddit_media_url'},
   {'id': '1rrqppu', 'reason': 'no_reddit_media_url'},
   {'id': '1rtoiok', 'reason': 'no_reddit_media_url'},
   {'id': '1s0ujtv', 'reason': 'no_reddit_media_url'},
   {'id': '1s39yao', 'reason': 'no_reddit_media_url'},
   {'id': '1s7ipj9', 'reason': 'no_reddit_media_url'},
   {'id': '1s7qboy', 'reason': 'no_reddit_media_url'},
   {'id': '1spu9k9', 'reason': 'no_reddit_media_url'},
 

In [10]:
move_stats = organize_downloads(
    input_dir="Media/media_files",  # where your downloader wrote files
    output_dir="Media",             # where Images/, Videos/, Gifs/ live
    strategy="move",
    conflict="move_existing",
    show_progress=True,
    dry_run=DRY_RUN_ORGANIZE,       # set True to preview
    prune_empty_galleries=True,     # remove empty src folders after moving
)

move_stats

Organizing media:   0%|          | 0/130 [00:00<?, ?file/s]

[SKIPPED→AlreadyMoved] 1mbe24q.mp4 → C:\Users\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1mbe24q.mp4
[SKIPPED→AlreadyMoved] 1qme7ed.mp4 → C:\Users\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1qme7ed.mp4
[SKIPPED→AlreadyMoved] 1rtwq7j.gif → C:\Users\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1rtwq7j.gif
[SKIPPED→AlreadyMoved] 1s2ii9h.jpeg → C:\Users\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1s2ii9h.jpeg
[SKIPPED→AlreadyMoved] 1sfdfi1.gif → C:\Users\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1sfdfi1.gif
[SKIPPED→AlreadyMoved] 1sgjtwl.gif → C:\Users\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1sgjtwl.gif
[SKIPPED→AlreadyMoved] 1sjtsz1.jpeg → C:\Users\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_

{'moved': 16,
 'copied': 0,
 'linked': 0,
 'skipped': 114,
 'unknown': 0,
 'dry_run': False,
 'strategy': 'move',
 'conflict': 'move_existing',
 'input_dir': 'C:\\Users\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files',
 'output_dir': 'C:\\Users\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media',
 'errors': [],
 'created_dirs': {'C:\\Users\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Images\\1ugz9ws',
  'C:\\Users\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Images\\1uh04dd'},
 'pruned_dirs': ['C:\\Users\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files\\1aee2qs',
  'C:\\Users\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files\\1mjphp6',
  'C:\\Users\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files\\1ppuwwk',
  'C:\\Users\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files\\1qz

# REDGIF DOWNLOADER

Downloads redgifs from external json folder in **Out/**

In [11]:
stats = process_external(
    media_json_dir=Path("Out/external"),
    media_out_dir=Path("Media/RedGiphys"),
    write_fail_csv_to=Path("__reports/redgif_report_" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
    write_links_csv_to=Path("__reports/external_links" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
    show_progress=True,
    dry_run=DRY_RUN_MEDIA,
    overwrite_downloads=False,
)

stats

Found 49 external post JSONs in Out\external


Scanning external posts:   0%|          | 0/49 [00:00<?, ?post/s]

[REDGIFS] id=1hvejqo -> 1hvejqo.mp4
[REDGIFS] id=1ubz3gl -> 1ubz3gl.mp4
[REDGIFS] id=1uf8rr0 -> 1uf8rr0.mp4
[REDGIFS] id=1uixwfb -> 1uixwfb.mp4
[REDGIFS] id=1ujuqky -> 1ujuqky.mp4
[REDGIFS] id=1unq6um -> 1unq6um.mp4
Saved external links to: C:\Users\minds\Desktop\Downloader and Reddit System\Saved-Reddit\__reports\external_links20260709-174509.csv


{'external_rows': [{'id': '1hvejqo',
   'link': 'https://www.redgifs.com/watch/doublechocolatemongrel',
   'domain': 'www.redgifs.com'},
  {'id': '1oyhfm4',
   'link': 'https://www.redgifs.com/watch/lustrousdisloyalsalamander',
   'domain': 'www.redgifs.com'},
  {'id': '1q6n5qv',
   'link': 'https://www.redgifs.com/watch/teemingutterzethuswasp',
   'domain': 'www.redgifs.com'},
  {'id': '1r40e0r',
   'link': 'https://www.redgifs.com/watch/hilariousspryeelelephant',
   'domain': 'www.redgifs.com'},
  {'id': '1rg3fpf',
   'link': 'https://www.redgifs.com/ifr/splendiddecisivenoddy/?2/',
   'domain': 'www.redgifs.com'},
  {'id': '1rg5m46',
   'link': 'https://www.redgifs.com/watch/tubbyhelplesspanther',
   'domain': 'www.redgifs.com'},
  {'id': '1rgmzbe',
   'link': 'https://www.redgifs.com/watch/menacingproperneonredguppy',
   'domain': 'www.redgifs.com'},
  {'id': '1rheclu',
   'link': 'https://www.redgifs.com/watch/weightyadolescentsulphurbutterfly',
   'domain': 'www.redgifs.com'},
  {

# CLOUDFLARE VERIFICATION & UPLOAD

In [12]:
raw_output = []
uploadsData = []

for mediaType in ["Images", "Videos", "Gifs", "RedGiphys"]:
    try:
        result = upload_media(
            input_path=Path("Media") / mediaType,  # where local files/galleries live
            r2_prefix=mediaType,                   # must match bucket prefix
            account_idx=ACCOUNT_INDEX - 1,         # <— choose which R2 credentials to use
            dry_run=DRY_RUN_CLOUDFLARE,            # preview vs. real upload
            overwrite=False,                       # don't overwrite existing objects
            check_only=CHECK_ONLY,                 # <— enable to just check existence
        )

        raw_output.append(result)
        # choose which list you want to visualize depending on mode
        if CHECK_ONLY:
            uploadsData.extend(result["exists"] + result["missing"])
        else:
            uploadsData.extend(result["planned"])

    except Exception as e:
        print(f"⚠️ Error on {mediaType}: {e}")
        continue

In [13]:
results_df = pd.DataFrame(uploadsData)
pd.set_option('display.max_rows', None)
results_df

,local,r2_key,bytes,content_type,status,account_index,bucket
0,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1s2ii9h.jpeg,111361,image/jpeg,planned,1,media-archive
1,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1sduirz.jpeg,184402,image/jpeg,planned,1,media-archive
2,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1sjtsz1.jpeg,47711,image/jpeg,planned,1,media-archive
3,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1srpirg.jpeg,420393,image/jpeg,planned,1,media-archive
4,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1ssgk5h.jpeg,119174,image/jpeg,planned,1,media-archive
5,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1suv8ta.jpeg,361128,image/jpeg,planned,1,media-archive
6,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1svycfq.jpeg,405355,image/jpeg,planned,1,media-archive
7,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1tmnnui.jpeg,55537,image/jpeg,planned,1,media-archive
8,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1uo00u7.jpeg,63575,image/jpeg,planned,1,media-archive
9,C:\Users\minds\Desktop\Downloader and Reddit S...,Images/1uqmt2x.png,1310981,image/png,planned,1,media-archive


# VERIFY UPLOAD

In [14]:
cats = ["Images", "RedGiphys", "Gifs", "Videos"]
all_rows = []

for cat in cats:
    try:
        res = audit_local_vs_r2(
            local_root=Path("Media") / cat,  # e.g., Media/Images
            r2_prefixes=[cat],               # matches your bucket key prefix
            account_indices=None,            # all accounts
            write_csv_to=None,               # (optional) per-cat CSV
            show_progress=True,
        )
        rows = res["rows"]
        for r in rows:
            r["category"] = cat
        all_rows.extend(rows)
    except Exception as e:
        print(f"⚠️ Error auditing {cat}: {e}")
        continue

audit_results = pd.DataFrame(all_rows)
pd.set_option("display.max_rows", None)
audit_results

Auditing: 100%|██████████| 2/2 [00:00<00:00, 1999.19file/s]


,local_rel,local_ext,all_expected_keys,matched,match_type,matched_prefix,matched_key,remote_ext,same_ext,matched_account_index,matched_bucket,note,category
0,1s2ii9h.jpeg,.jpeg,Images/1s2ii9h.jpeg,True,exact,Images,Images/1s2ii9h.jpeg,.jpeg,True,1,media-archive,exact_match,Images
1,1sduirz.jpeg,.jpeg,Images/1sduirz.jpeg,True,exact,Images,Images/1sduirz.jpeg,.jpeg,True,1,media-archive,exact_match,Images
2,1sjtsz1.jpeg,.jpeg,Images/1sjtsz1.jpeg,True,exact,Images,Images/1sjtsz1.jpeg,.jpeg,True,1,media-archive,exact_match,Images
3,1srpirg.jpeg,.jpeg,Images/1srpirg.jpeg,True,exact,Images,Images/1srpirg.jpeg,.jpeg,True,1,media-archive,exact_match,Images
4,1ssgk5h.jpeg,.jpeg,Images/1ssgk5h.jpeg,True,exact,Images,Images/1ssgk5h.jpeg,.jpeg,True,1,media-archive,exact_match,Images
5,1suv8ta.jpeg,.jpeg,Images/1suv8ta.jpeg,True,exact,Images,Images/1suv8ta.jpeg,.jpeg,True,1,media-archive,exact_match,Images
6,1svycfq.jpeg,.jpeg,Images/1svycfq.jpeg,True,exact,Images,Images/1svycfq.jpeg,.jpeg,True,1,media-archive,exact_match,Images
7,1tmnnui.jpeg,.jpeg,Images/1tmnnui.jpeg,True,exact,Images,Images/1tmnnui.jpeg,.jpeg,True,1,media-archive,exact_match,Images
8,1uo00u7.jpeg,.jpeg,Images/1uo00u7.jpeg,True,exact,Images,Images/1uo00u7.jpeg,.jpeg,True,1,media-archive,exact_match,Images
9,1uqmt2x.png,.png,Images/1uqmt2x.png,True,exact,Images,Images/1uqmt2x.png,.png,True,1,media-archive,exact_match,Images


In [15]:
if DRY_RUN_FINAL or (False in audit_results["matched"].value_counts().keys()):
    print(final_df.head(20))
    print("Dry run enabled; ordered_posts not changed")
else:
    print("Updating ordered_posts.csv with new posts...")
    final_df.to_csv("ordered_posts.csv", index=False)

Updating ordered_posts.csv with new posts...
